# 20 — Restate Runtime: Durable Agent Execution

This notebook demonstrates the **unified Restate durable execution layer** — the
result of merging the old `distributed/` package and `catalog/_temporal/` into
`integrations/runtime/restate/`.

### What changed

| Old | New |
|---|---|
| `distributed/policies.py` | `integrations/runtime/restate/policies.py` |
| `distributed/client.py` (RestateClient) | `integrations/runtime/restate/client.py` (RestateWorkflowClient) |
| `distributed/workflow.py` (AgentWorkflow) | `integrations/runtime/restate/workflows.py` (all 3 workflows) |
| `distributed/activities.py` | `integrations/runtime/restate/activities.py` (merged) |
| `catalog/_temporal/client.py` (TemporalClient) | same `RestateWorkflowClient` (drop-in) |
| `catalog/_temporal/workflows.py` | `integrations/runtime/restate/workflows.py` |
| `temporalio` dependency | **removed** |

### Sections

| # | Section | Infrastructure needed |
|---|---|---|
| 1 | Package structure | none |
| 2 | `ToolPolicy` — per-tool execution governance | none |
| 3 | `RestateWorkflowClient` — connect and inspect | none (mocked) |
| 4 | `Activities` — DI configuration | none |
| 5 | `LocalRuntime` — run an agent in-process | none |
| 6 | Durable agent workflow — journey through Restate | Restate server |
| 7 | HITL promise suspension flow | Restate server |
| 8 | Pipeline workflow | Restate server |

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

## 1. Package Structure

All durable execution code now lives in one place.

In [2]:
import importlib, pkgutil

pkg = importlib.import_module("raavan.integrations.runtime.restate")
print("Public exports:", pkg.__all__)

# Walk the submodule files
import raavan.integrations.runtime.restate as restate_pkg
import pathlib

base = pathlib.Path(restate_pkg.__file__).parent
for f in sorted(base.iterdir()):
    if f.suffix == ".py" and not f.name.startswith("__"):
        print(f"  {f.name}")

Public exports: ['RestateRuntime', 'RestateWorkflowClient', 'ToolPolicy', 'derive_policy_from_tool']
  activities.py
  app.py
  client.py
  policies.py
  runtime.py
  worker.py
  workflows.py


## 2. `ToolPolicy` — Per-Tool Execution Governance

`ToolPolicy` is a frozen dataclass that controls **how** each tool runs inside a durable workflow:
- `timeout` — max seconds before the activity fails
- `needs_idempotency` — generate a stable `ctx.uuid()` key (prevents double-writes)
- `requires_approval` — suspend and wait for HITL approval before executing
- `is_hitl_input` — this tool is human-input (ask_human); prompts the user
- `max_retries` — Restate retry budget on transient failure

The static `TOOL_POLICIES` table covers known tools. Unknown tools fall back to `derive_policy_from_tool(tool)` which reads `tool.risk` and `tool.hitl_mode`.

In [3]:
from raavan.integrations.runtime.restate.policies import (
    ToolPolicy, TOOL_POLICIES, get_policy, derive_policy_from_tool
)

# ── Static table lookup ──────────────────────────────────────────────────
print("=== Known tool policies ===")
for name, policy in TOOL_POLICIES.items():
    flags = []
    if policy.is_hitl_input:      flags.append("hitl_input")
    if policy.requires_approval:  flags.append("approval_gate")
    if policy.needs_idempotency:  flags.append("idempotent")
    print(f"  {name:20s}  timeout={policy.timeout:5.0f}s  {', '.join(flags) or 'standard'}")

print()
# ── Unknown tool falls back to default ──────────────────────────────────
p = get_policy("some_new_tool")
print(f"Unknown tool policy: {p}")

# ── Frozen — mutation is rejected ───────────────────────────────────────
try:
    p.timeout = 999  # type: ignore[misc]
except AttributeError as e:
    print(f"\nFrozen ✓  ({e})")

=== Known tool policies ===
  ask_human             timeout=  300s  hitl_input
  send_email            timeout=   30s  approval_gate, idempotent
  manage_tasks          timeout=   15s  standard
  web_surfer            timeout=  120s  standard
  code_interpreter      timeout=  300s  standard
  file_manager          timeout=   60s  standard
  calculator            timeout=   10s  standard
  data_visualizer       timeout=   30s  standard

Unknown tool policy: ToolPolicy(timeout=30.0, needs_idempotency=False, requires_approval=False, is_hitl_input=False, large_payload=False, max_retries=1)

Frozen ✓  (cannot assign to field 'timeout')


## 3. `RestateWorkflowClient` — Unified Workflow HTTP Client

`RestateWorkflowClient` replaces both the old `TemporalClient` and the old `RestateClient`.
It speaks directly to the **Restate ingress** (`/invoke/WorkflowName/WorkflowId/handler`)
and the **admin API** (`/deployments`) over a persistent `httpx.AsyncClient` connection.

| Old symbol | New symbol |
|---|---|
| `TemporalClient` | `RestateWorkflowClient` |
| `app.state.temporal` | `app.state.workflow_client` |
| Temporal TaskQueue | `WorkflowId` (URL path segment) |

The three workflow entry-points:

```
start_pipeline_workflow(pipeline_id, steps, context) → AgentWorkflowHandle
start_chain_workflow(chain_id, code, inputs, context) → AgentWorkflowHandle
start_agent_workflow(conversation_id, message, context) → AgentWorkflowHandle
```

All three are **durable** — Restate persists every activity result so a crash resumes
from the last completed step rather than re-executing from the top.

In [5]:
from unittest.mock import AsyncMock, MagicMock, patch
from raavan.integrations.runtime.restate.client import RestateWorkflowClient

# Build a client without live infrastructure
client = RestateWorkflowClient(
    ingress_url="http://localhost:8080",
    admin_url="http://localhost:9070",
)

print("Client created:", client)
print(f"  ingress : {client._ingress_url}")
print(f"  admin   : {client._admin_url}")
print()

# ── Mock the underlying HTTP call so we can exercise the path without infra ──
fake_response = MagicMock()
fake_response.raise_for_status = MagicMock()
fake_response.json.return_value = {"id": "conv-001", "status": "RUNNING"}

with patch.object(client, "_invoke", new=AsyncMock(return_value={"wid": "conv-001"})):
    import asyncio

    async def demo():
        handle = await client.start_agent_workflow(
            conversation_id="conv-001",
            user_message="Analyse the Q3 sales report",
            context={"thread_id": "thread-abc"},
        )
        print("Workflow started →", handle)

        status = await client.query_workflow("AgentWorkflow", "conv-001", "status")
        print("Query  →", status)

    await demo()

Client created: <raavan.integrations.runtime.restate.client.RestateWorkflowClient object at 0x00000182A65FDDD0>
  ingress : http://localhost:8080
  admin   : http://localhost:9070



TypeError: RestateWorkflowClient.start_agent_workflow() got an unexpected keyword argument 'conversation_id'

## 4. Activities DI — Wiring Dependencies at Worker Startup

All activities live in `integrations/runtime/restate/activities.py` and share state through
a single `configure()` call at worker start. This avoids passing fat context dicts
through every activity signature.

```python
# worker.py (simplified)
from raavan.integrations.runtime.restate.activities import configure

configure(
    streaming      = nats_bridge,       # NATSBridge — fan-out SSE events to clients
    model_client   = openai_client,     # BaseModelClient — LLM calls
    tools          = tool_registry,     # dict[str, BaseTool] — tool catalog
    redis_memory   = redis_memory,      # RedisMemory — per-conversation history
    catalog        = capability_catalog,# CapabilityCatalog — skill + connector lookup
    data_store     = data_ref_store,    # DataRefStore — inter-step data refs
    chain_runtime  = chain_runtime,     # ChainRuntime — execute LLM-written code chains
)
```

The activities themselves are thin wrappers. Each is decorated with
`@activity.defn(name="...")` so Restate can route invocations by name.
Restate calls them in-process inside the worker; the durable journal
is stored in Restate's own storage layer.

In [ ]:
import inspect
from raavan.integrations.runtime.restate import activities

# Show every public function and its signature
print("=== Activity functions ===")
for name, fn in inspect.getmembers(activities, inspect.iscoroutinefunction):
    if name.startswith("_"):
        continue
    sig = inspect.signature(fn)
    print(f"  async {name}{sig}")

print()
print("=== Module-level DI slots ===")
slots = ["_streaming", "_model_client", "_tools", "_redis_memory",
         "_catalog", "_data_store", "_chain_runtime"]
for slot in slots:
    val = getattr(activities, slot, "<unset>")
    print(f"  {slot:20s}  {'✓ set' if val is not None else '— None (call configure() first)'}")

## 5. `LocalRuntime` — Run an Agent Without Any Infrastructure

`LocalRuntime` executes the full ReAct loop in-process with no Restate, no NATS,
no Postgres needed. It is the default choice for:
- Notebooks and exploration
- Unit / integration tests
- Single-user or development deployments

The same `BaseAgent.run()` entrypoint is used regardless of which runtime is
configured — the runtime is a strategy injected at construction time.

In [6]:
import os, asyncio
from unittest.mock import AsyncMock, MagicMock, patch

# --- mock LLM so demo runs without an API key ---
mock_llm_response = MagicMock()
mock_llm_response.content = [MagicMock(type="text", text="The capital of France is Paris.")]
mock_llm_response.stop_reason = "end_turn"
mock_llm_response.tool_calls = []

from raavan.core.runtime import LocalRuntime
from raavan.core.agents.react_agent import ReActAgent
from raavan.core.memory import UnboundedMemory

async def run_local_agent():
    memory = UnboundedMemory()

    with patch(
        "raavan.integrations.llm.openai.openai_client.OpenAIClient.generate",
        new=AsyncMock(return_value=mock_llm_response),
    ):
        from raavan.integrations.llm.openai.openai_client import OpenAIClient

        llm = OpenAIClient(api_key="sk-fake", model="gpt-4o")
        runtime = LocalRuntime()

        agent = ReActAgent(
            name="demo-agent",
            model_client=llm,
            memory=memory,
            runtime=runtime,
            tools=[],
            system_prompt="You are a concise assistant.",
        )

        response = await agent.run("What is the capital of France?")
        print("Agent response:", response)
        print()
        messages = await memory.get_messages()
        print(f"Messages in memory: {len(messages)}")
        for m in messages:
            print(f"  [{type(m).__name__}]", str(m.content)[:80])

asyncio.run(run_local_agent())

RuntimeError: asyncio.run() cannot be called from a running event loop

## 6. Durable Agent Workflow Journey

When `GrpcRuntime` (or the `RestateRuntime` adapter) is used, `agent.run()` submits
an `AgentWorkflow` to Restate instead of running the loop locally.

The durable execution path:

```
user message
    │
    ▼
FastAPI POST /chat
    │ RestateWorkflowClient.start_agent_workflow()
    ▼
Restate Ingress (HTTP 2xx, workflow durably accepted)
    │
    ▼
Worker picks up AgentWorkflow
    │
    ├─► restore_memory()      ← replay Redis history
    │
    └─► [ReAct loop, each iteration:]
            ├─► do_llm_call()       ← Restate persists result
            │       │ stop_reason == "tool_use"?
            │       ▼
            ├─► do_tool_exec()      ← Restate persists result
            │       │ result ready?
            │       ▼
            ├─► persist_message()   ← stores to Redis + Postgres
            ├─► publish_event()     ← NATS → live_stream → SSE → browser
            │
            └─► (loop until end_turn or max_iterations)
```

If the worker crashes between any two activities, Restate replays the journal
and resumes from the **last committed activity** — no duplicated LLM calls.

In [2]:
from IPython.display import HTML

html = """
<style>
  @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700;800&display=swap');

  .af-wrap {
    font-family: 'Inter', sans-serif;
    background: #ffffff;
    border-radius: 16px;
    padding: 32px 36px;
    max-width: 1120px;
    margin: 0 auto;
    box-shadow: 0 4px 32px rgba(0,0,0,0.10);
    color: #1a1a2e;
  }
  .af-title {
    font-size: 22px; font-weight: 800; text-align: center;
    color: #1a1a2e; margin: 0 0 4px 0; letter-spacing: -0.5px;
  }
  .af-subtitle {
    font-size: 13px; text-align: center; color: #6b7280; margin: 0 0 28px 0;
  }
  /* ── Section labels ── */
  .af-section-label {
    font-size: 11px; font-weight: 700; letter-spacing: 1.2px;
    text-transform: uppercase; color: #9ca3af; margin-bottom: 10px;
    padding-left: 4px;
  }
  /* ── Row layout ── */
  .af-row {
    display: flex; align-items: center; gap: 0; margin-bottom: 6px;
  }
  /* ── Boxes ── */
  .af-box {
    border-radius: 10px; padding: 10px 14px; text-align: center;
    flex: 1; min-width: 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.07);
  }
  .af-box .icon { font-size: 22px; line-height: 1; }
  .af-box .label { font-size: 11px; font-weight: 700; margin-top: 4px; letter-spacing: 0.2px; }
  .af-box .sub   { font-size: 10px; color: #6b7280; margin-top: 2px; }

  /* colour variants */
  .box-blue   { background: #eff6ff; border: 2px solid #3b82f6; }
  .box-blue .label   { color: #1d4ed8; }
  .box-violet { background: #f5f3ff; border: 2px solid #7c3aed; }
  .box-violet .label { color: #5b21b6; }
  .box-indigo { background: #eef2ff; border: 2px solid #4f46e5; }
  .box-indigo .label { color: #3730a3; }
  .box-teal   { background: #f0fdfa; border: 2px solid #0d9488; }
  .box-teal .label   { color: #0f766e; }
  .box-amber  { background: #fffbeb; border: 2px solid #d97706; }
  .box-amber .label  { color: #92400e; }
  .box-rose   { background: #fff1f2; border: 2px solid #e11d48; }
  .box-rose .label   { color: #9f1239; }
  .box-emerald{ background: #ecfdf5; border: 2px solid #059669; }
  .box-emerald .label{ color: #065f46; }
  .box-slate  { background: #f8fafc; border: 2px solid #64748b; }
  .box-slate .label  { color: #334155; }
  .box-orange { background: #fff7ed; border: 2px solid #ea580c; }
  .box-orange .label { color: #9a3412; }
  .box-sky    { background: #f0f9ff; border: 2px solid #0284c7; }
  .box-sky .label    { color: #0c4a6e; }

  /* ── Arrow ── */
  .af-arrow {
    flex: 0 0 auto; width: 28px; text-align: center;
    font-size: 16px; color: #9ca3af; position: relative;
  }
  .af-arrow .step-num {
    position: absolute; top: -9px; left: 50%;
    transform: translateX(-50%);
    background: #1a1a2e; color: #fff;
    font-size: 9px; font-weight: 700;
    width: 16px; height: 16px; border-radius: 50%;
    display: flex; align-items: center; justify-content: center;
    line-height: 1;
  }
  .af-down-arrow {
    text-align: center; font-size: 18px; color: #9ca3af;
    height: 22px; line-height: 22px; margin: 0 0;
  }
  /* ── Worker loop box ── */
  .af-loop-box {
    border: 2px dashed #7c3aed; border-radius: 12px;
    padding: 16px 20px; margin: 4px 0 4px 0;
    background: #faf5ff;
  }
  .af-loop-title {
    font-size: 11px; font-weight: 700; color: #5b21b6;
    letter-spacing: 0.8px; text-transform: uppercase; margin-bottom: 12px;
    text-align: center;
  }
  .af-loop-row {
    display: flex; align-items: center; justify-content: center; gap: 0;
    margin-bottom: 8px; overflow-x: auto;
  }
  .af-loop-box .af-box { flex: 0 0 auto; width: 115px; }
  /* ── HITL section ── */
  .af-hitl {
    border: 2px solid #f59e0b; border-radius: 12px;
    background: #fffbeb; padding: 16px 20px; margin-top: 20px;
  }
  .af-hitl-title {
    font-size: 12px; font-weight: 800; color: #b45309;
    letter-spacing: 0.6px; text-transform: uppercase; margin-bottom: 12px;
  }
  .af-hitl-row {
    display: flex; align-items: center; gap: 0; flex-wrap: nowrap;
    overflow-x: auto;
  }
  .af-hitl .af-box { flex: 0 0 auto; width: 120px; }
  /* ── Crash guarantee box ── */
  .af-guarantee {
    margin-top: 20px; border-radius: 12px; padding: 14px 20px;
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
    color: #e2e8f0; display: flex; align-items: center; gap: 14px;
  }
  .af-guarantee .big-icon { font-size: 32px; flex-shrink: 0; }
  .af-guarantee .title    { font-size: 13px; font-weight: 700; color: #fff; }
  .af-guarantee .body     { font-size: 11px; color: #94a3b8; margin-top: 2px; }
  /* ── Divider ── */
  .af-divider {
    border: none; border-top: 1px solid #e5e7eb; margin: 18px 0;
  }
  /* response row reversed arrow */
  .af-arrow-left { color: #9ca3af; }
</style>

<div class="af-wrap">

  <p class="af-title">🤖 How the Agent Framework Handles Your Message</p>
  <p class="af-subtitle">End-to-end in under 5 min — every box is a real service or component</p>

  <!-- ═══════════════════════ REQUEST PATH ═══════════════════════ -->
  <div class="af-section-label">① REQUEST PATH — your message travels here</div>
  <div class="af-row">
    <div class="af-box box-blue">
      <div class="icon">🌐</div>
      <div class="label">Browser</div>
      <div class="sub">React UI</div>
    </div>
    <div class="af-arrow"><span class="step-num">1</span>→</div>
    <div class="af-box box-sky">
      <div class="icon">⚡</div>
      <div class="label">Next.js BFF</div>
      <div class="sub">Proxy + SSE</div>
    </div>
    <div class="af-arrow"><span class="step-num">2</span>→</div>
    <div class="af-box box-violet">
      <div class="icon">🔌</div>
      <div class="label">FastAPI</div>
      <div class="sub">Registers SSE channel</div>
    </div>
    <div class="af-arrow"><span class="step-num">3</span>→</div>
    <div class="af-box box-indigo">
      <div class="icon">💾</div>
      <div class="label">Restate</div>
      <div class="sub">Durably accepted</div>
    </div>
    <div class="af-arrow"><span class="step-num">4</span>→</div>
    <div class="af-box box-teal">
      <div class="icon">⚙️</div>
      <div class="label">Worker</div>
      <div class="sub">Picks up workflow</div>
    </div>
  </div>

  <div class="af-down-arrow">↓</div>

  <!-- ═══════════════════════ WORKER INTERNALS ═══════════════════════ -->
  <div class="af-loop-box">
    <div class="af-loop-title">🔄 ReAct Loop — runs inside Worker (each turn is durably journalled)</div>

    <div class="af-loop-row">
      <div class="af-box box-amber">
        <div class="icon">🧠</div>
        <div class="label">Redis Memory</div>
        <div class="sub">Restores history</div>
      </div>
      <div class="af-arrow"><span class="step-num">5</span>→</div>
      <div class="af-box box-rose">
        <div class="icon">🤖</div>
        <div class="label">LLM Call</div>
        <div class="sub">GPT-4o thinks</div>
      </div>
      <div class="af-arrow"><span class="step-num">6</span>→</div>
      <div class="af-box box-orange">
        <div class="icon">🔧</div>
        <div class="label">Tool Exec</div>
        <div class="sub">web_surfer, code…</div>
      </div>
      <div class="af-arrow"><span class="step-num">7</span>→</div>
      <div class="af-box box-emerald">
        <div class="icon">💿</div>
        <div class="label">Persist</div>
        <div class="sub">Redis + Postgres</div>
      </div>
      <div class="af-arrow"><span class="step-num">8</span>→</div>
      <div class="af-box box-slate">
        <div class="icon">📡</div>
        <div class="label">NATS Publish</div>
        <div class="sub">Broadcasts tokens</div>
      </div>
    </div>
  </div>

  <div class="af-down-arrow">↓</div>

  <!-- ═══════════════════════ RESPONSE PATH ═══════════════════════ -->
  <div class="af-section-label">② RESPONSE PATH — tokens stream back to your screen</div>
  <div class="af-row">
    <div class="af-box box-slate">
      <div class="icon">📡</div>
      <div class="label">NATS Bus</div>
      <div class="sub">Event fan-out</div>
    </div>
    <div class="af-arrow"><span class="step-num">9</span>→</div>
    <div class="af-box box-violet">
      <div class="icon">🌊</div>
      <div class="label">SSE Bridge</div>
      <div class="sub">FastAPI pushes</div>
    </div>
    <div class="af-arrow"><span class="step-num">10</span>→</div>
    <div class="af-box box-sky">
      <div class="icon">⚡</div>
      <div class="label">Next.js BFF</div>
      <div class="sub">Forwards stream</div>
    </div>
    <div class="af-arrow"><span class="step-num">11</span>→</div>
    <div class="af-box box-blue">
      <div class="icon">✨</div>
      <div class="label">Browser</div>
      <div class="sub">Renders tokens live</div>
    </div>
  </div>

  <hr class="af-divider"/>

  <!-- ═══════════════════════ HITL SECTION ═══════════════════════ -->
  <div class="af-hitl">
    <div class="af-hitl-title">⏸️ What if a tool needs human approval? (HITL flow)</div>
    <div class="af-hitl-row">
      <div class="af-box box-orange">
        <div class="icon">🔧</div>
        <div class="label">Risky Tool</div>
        <div class="sub">send_email etc.</div>
      </div>
      <div class="af-arrow">→</div>
      <div class="af-box box-indigo">
        <div class="icon">⏸️</div>
        <div class="label">Restate Pauses</div>
        <div class="sub">Promise created</div>
      </div>
      <div class="af-arrow">→</div>
      <div class="af-box box-blue">
        <div class="icon">🙋</div>
        <div class="label">UI Prompts You</div>
        <div class="sub">Approval card shown</div>
      </div>
      <div class="af-arrow">→</div>
      <div class="af-box box-emerald">
        <div class="icon">✅</div>
        <div class="label">You Approve</div>
        <div class="sub">POST /hitl/respond</div>
      </div>
      <div class="af-arrow">→</div>
      <div class="af-box box-teal">
        <div class="icon">▶️</div>
        <div class="label">Workflow Resumes</div>
        <div class="sub">From exact pause point</div>
      </div>
    </div>
  </div>

  <!-- ═══════════════════════ CRASH GUARANTEE ═══════════════════════ -->
  <div class="af-guarantee">
    <div class="big-icon">🛡️</div>
    <div>
      <div class="title">Crash-Safe by Default</div>
      <div class="body">
        If the Worker crashes after step 6 (tool exec) but before step 7 (persist),
        Restate replays the durable journal and resumes from step 6's saved result.
        The LLM is <strong style="color:#fff">never called twice</strong>.
        Your conversation state is <strong style="color:#fff">never lost</strong>.
      </div>
    </div>
  </div>

</div>
"""

HTML(html)

In [1]:
from IPython.display import HTML

HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800;900&display=swap');
* { box-sizing: border-box; margin: 0; padding: 0; }

.ar-root {
  font-family: 'Inter', sans-serif;
  background: #0f172a;
  border-radius: 20px;
  padding: 30px 32px 28px;
  max-width: 1120px;
  margin: 0 auto;
  box-shadow: 0 12px 60px rgba(0,0,0,0.5);
}

/* ── Header ──────────────────────────────────────────────────── */
.ar-hdr { text-align: center; margin-bottom: 22px; }
.ar-hdr-title {
  font-size: 21px; font-weight: 900; letter-spacing: -0.5px;
  background: linear-gradient(135deg, #e2e8f0 0%, #818cf8 60%, #38bdf8 100%);
  -webkit-background-clip: text; -webkit-text-fill-color: transparent;
}
.ar-hdr-sub {
  font-size: 11px; color: #64748b; margin-top: 5px; letter-spacing: 0.3px;
}
.ar-hdr-pill {
  display: inline-block; background: #1e293b; border: 1px solid #334155;
  border-radius: 20px; padding: 3px 12px; margin: 8px 4px 0;
  font-size: 10px; font-weight: 600; color: #94a3b8;
}

/* ── 4-zone grid ─────────────────────────────────────────────── */
.ar-zones {
  display: grid;
  grid-template-columns: 1fr 36px 1fr 36px 1fr 36px 1fr;
  gap: 0;
  margin-bottom: 10px;
  align-items: stretch;
}
.ar-zone {
  border-radius: 14px; padding: 14px 14px 12px;
  display: flex; flex-direction: column; gap: 8px;
}
.z-client  { background: #0c1a35; border: 1.5px solid #3b82f6; }
.z-api     { background: #160c2e; border: 1.5px solid #7c3aed; }
.z-exec    { background: #0d1634; border: 1.5px solid #4f46e5; }
.z-infra   { background: #0b1f1e; border: 1.5px solid #0d9488; }

.ar-zone-label {
  font-size: 9px; font-weight: 800; letter-spacing: 1.8px;
  text-transform: uppercase; margin-bottom: 2px;
}
.z-client .ar-zone-label  { color: #60a5fa; }
.z-api    .ar-zone-label  { color: #a78bfa; }
.z-exec   .ar-zone-label  { color: #818cf8; }
.z-infra  .ar-zone-label  { color: #2dd4bf; }

/* ── Actor cards ─────────────────────────────────────────────── */
.ar-actor {
  background: #1e293b; border-radius: 10px;
  padding: 10px 12px; border-left: 3px solid transparent;
}
.ar-actor-head {
  display: flex; align-items: center; gap: 8px; margin-bottom: 6px;
}
.ar-actor-icon  { font-size: 18px; flex-shrink: 0; line-height: 1; }
.ar-actor-name  { font-size: 11px; font-weight: 700; color: #e2e8f0; line-height: 1.2; }
.ar-actor-role  { font-size: 9px; color: #64748b; margin-top: 1px; }

/* mailbox tag row */
.ar-tags { display: flex; flex-wrap: wrap; gap: 3px; }
.ar-tag  {
  font-size: 8px; font-weight: 700; padding: 2px 5px;
  border-radius: 4px; white-space: nowrap; letter-spacing: 0.2px;
}
.tag-recv { background: #0f2b4a; color: #38bdf8; border: 1px solid #0369a1; }
.tag-send { background: #0f2e1a; color: #4ade80; border: 1px solid #166534; }
.tag-store{ background: #1e1b3a; color: #a78bfa; border: 1px solid #4f46e5; }

/* actor colour accents */
.a-blue   { border-left-color: #3b82f6; }
.a-sky    { border-left-color: #0ea5e9; }
.a-violet { border-left-color: #7c3aed; }
.a-purple { border-left-color: #9333ea; }
.a-indigo { border-left-color: #4f46e5; }
.a-teal   { border-left-color: #0d9488; }
.a-amber  { border-left-color: #d97706; }
.a-green  { border-left-color: #059669; }

/* ── Zone connectors ─────────────────────────────────────────── */
.ar-conn {
  display: flex; flex-direction: column; align-items: center;
  justify-content: center; gap: 5px; padding: 4px 0;
}
.ar-conn-msg {
  font-size: 8px; font-weight: 700; color: #475569; text-align: center;
  background: #1e293b; border: 1px solid #334155; border-radius: 5px;
  padding: 3px 4px; line-height: 1.4; white-space: nowrap;
}
.ar-conn-arrow { font-size: 14px; color: #475569; line-height: 1; }
.ar-conn-back  { font-size: 8px; color: #475569; font-weight: 600; text-align:center; }

/* ── Infra actors row (inside z-infra) ───────────────────────── */
.ar-infra-grid {
  display: grid; grid-template-columns: 1fr 1fr; gap: 6px;
  height: 100%;
}

/* ── Separator ───────────────────────────────────────────────── */
.ar-sep {
  height: 1px; background: #1e293b; margin: 8px 0;
}

/* ── Flows section ───────────────────────────────────────────── */
.ar-flows {
  display: grid; grid-template-columns: 1fr 1fr; gap: 10px;
  margin-bottom: 10px;
}
.ar-flow-card {
  border-radius: 12px; padding: 16px 16px 12px;
}
.fc-normal { background: #071d12; border: 1.5px solid #16a34a; }
.fc-hitl   { background: #1a1200; border: 1.5px solid #d97706; }
.ar-flow-title {
  font-size: 10px; font-weight: 800; letter-spacing: 1px;
  text-transform: uppercase; margin-bottom: 12px;
}
.fc-normal .ar-flow-title { color: #4ade80; }
.fc-hitl   .ar-flow-title { color: #fbbf24; }

.ar-step {
  display: flex; align-items: flex-start; gap: 8px; margin-bottom: 8px;
}
.ar-step:last-child { margin-bottom: 0; }
.ar-step-num {
  width: 18px; height: 18px; border-radius: 50%; flex-shrink: 0;
  font-size: 9px; font-weight: 800; color: #0f172a;
  display: flex; align-items: center; justify-content: center;
  margin-top: 1px;
}
.fc-normal .ar-step-num { background: #4ade80; }
.fc-hitl   .ar-step-num { background: #fbbf24; }
.ar-step-body { flex: 1; }
.ar-step-who {
  font-size: 9px; font-weight: 700;
  display: inline-block; padding: 1px 5px; border-radius: 3px;
  margin-right: 4px; margin-bottom: 2px;
}
.fc-normal .ar-step-who { background: #0f2e1a; color: #4ade80; }
.fc-hitl   .ar-step-who { background: #2b1a00; color: #fbbf24; }
.ar-step-text { font-size: 10px; color: #cbd5e1; line-height: 1.5; }

/* ── Crash banner ─────────────────────────────────────────────── */
.ar-crash {
  background: linear-gradient(90deg, #1e293b 0%, #1a1a2e 100%);
  border: 1px solid #334155; border-radius: 12px;
  padding: 14px 20px; display: flex; align-items: center; gap: 14px;
  margin-top: 2px;
}
.ar-crash-icon { font-size: 28px; flex-shrink: 0; }
.ar-crash-title { font-size: 12px; font-weight: 800; color: #e2e8f0; }
.ar-crash-body  { font-size: 10px; color: #64748b; margin-top: 2px; line-height: 1.5; }
.hi { color: #f8fafc; font-weight: 700; }
</style>

<div class="ar-root">
  <!-- ════════ HEADER ════════ -->
  <div class="ar-hdr">
    <div class="ar-hdr-title">🏗️ Agent Framework — Actor Model &amp; System Architecture</div>
    <div class="ar-hdr-sub">Every box is an isolated actor · Arrows are the only way actors communicate · Zones are deployment boundaries</div>
    <span class="ar-hdr-pill">📬 Receives</span>
    <span class="ar-hdr-pill">📤 Sends</span>
    <span class="ar-hdr-pill">🗃️ Persists</span>
  </div>

  <!-- ════════ ZONE GRID ════════ -->
  <div class="ar-zones">

    <!-- ── ZONE 1: CLIENT ── -->
    <div class="ar-zone z-client">
      <div class="ar-zone-label">① Client Tier</div>

      <div class="ar-actor a-blue">
        <div class="ar-actor-head">
          <div class="ar-actor-icon">🌐</div>
          <div><div class="ar-actor-name">Browser</div><div class="ar-actor-role">React UI</div></div>
        </div>
        <div class="ar-tags">
          <span class="ar-tag tag-recv">📬 SSE tokens</span>
          <span class="ar-tag tag-recv">📬 tool_approval</span>
          <span class="ar-tag tag-send">📤 POST /chat</span>
          <span class="ar-tag tag-send">📤 POST /hitl</span>
        </div>
      </div>

      <div class="ar-actor a-sky">
        <div class="ar-actor-head">
          <div class="ar-actor-icon">⚡</div>
          <div><div class="ar-actor-name">Next.js BFF</div><div class="ar-actor-role">Proxy + SSE pipe</div></div>
        </div>
        <div class="ar-tags">
          <span class="ar-tag tag-recv">📬 browser req</span>
          <span class="ar-tag tag-send">📤 proxy to API</span>
          <span class="ar-tag tag-send">📤 fwd SSE stream</span>
        </div>
      </div>
    </div>

    <!-- ── CONNECTOR 1 ── -->
    <div class="ar-conn">
      <div class="ar-conn-msg">HTTP POST</div>
      <div class="ar-conn-arrow">⟶</div>
      <div class="ar-conn-msg">SSE stream</div>
      <div class="ar-conn-arrow">⟵</div>
    </div>

    <!-- ── ZONE 2: API ── -->
    <div class="ar-zone z-api">
      <div class="ar-zone-label">② API Tier</div>

      <div class="ar-actor a-violet">
        <div class="ar-actor-head">
          <div class="ar-actor-icon">🔌</div>
          <div><div class="ar-actor-name">FastAPI Server</div><div class="ar-actor-role">Routes + auth + DI</div></div>
        </div>
        <div class="ar-tags">
          <span class="ar-tag tag-recv">📬 POST /chat</span>
          <span class="ar-tag tag-recv">📬 POST /hitl/respond</span>
          <span class="ar-tag tag-send">📤 start_workflow</span>
          <span class="ar-tag tag-send">📤 resolve_promise</span>
        </div>
      </div>

      <div class="ar-actor a-purple">
        <div class="ar-actor-head">
          <div class="ar-actor-icon">🌊</div>
          <div><div class="ar-actor-name">WebHITL Bridge</div><div class="ar-actor-role">NATS → SSE fanout</div></div>
        </div>
        <div class="ar-tags">
          <span class="ar-tag tag-recv">📬 NATS events</span>
          <span class="ar-tag tag-send">📤 push SSE per conv</span>
        </div>
      </div>
    </div>

    <!-- ── CONNECTOR 2 ── -->
    <div class="ar-conn">
      <div class="ar-conn-msg">HTTP<br>invoke</div>
      <div class="ar-conn-arrow">⟶</div>
      <div class="ar-conn-msg">NATS<br>events</div>
      <div class="ar-conn-arrow">⟵</div>
    </div>

    <!-- ── ZONE 3: DURABLE EXEC ── -->
    <div class="ar-zone z-exec">
      <div class="ar-zone-label">③ Durable Execution Tier</div>

      <div class="ar-actor a-indigo">
        <div class="ar-actor-head">
          <div class="ar-actor-icon">💾</div>
          <div><div class="ar-actor-name">Restate Cluster</div><div class="ar-actor-role">Journal + promises</div></div>
        </div>
        <div class="ar-tags">
          <span class="ar-tag tag-recv">📬 workflow start</span>
          <span class="ar-tag tag-recv">📬 promise resolve</span>
          <span class="ar-tag tag-send">📤 dispatch task</span>
          <span class="ar-tag tag-store">🗃️ durable journal</span>
        </div>
      </div>

      <div class="ar-actor a-teal">
        <div class="ar-actor-head">
          <div class="ar-actor-icon">⚙️</div>
          <div><div class="ar-actor-name">Worker + ReAct Loop</div><div class="ar-actor-role">Agent activities</div></div>
        </div>
        <div class="ar-tags">
          <span class="ar-tag tag-recv">📬 workflow tasks</span>
          <span class="ar-tag tag-send">📤 LLM call</span>
          <span class="ar-tag tag-send">📤 tool exec</span>
          <span class="ar-tag tag-send">📤 NATS publish</span>
          <span class="ar-tag tag-store">🗃️ Redis + PG</span>
        </div>
      </div>
    </div>

    <!-- ── CONNECTOR 3 ── -->
    <div class="ar-conn">
      <div class="ar-conn-msg">async<br>calls</div>
      <div class="ar-conn-arrow">⟶</div>
      <div class="ar-conn-msg">results</div>
      <div class="ar-conn-arrow">⟵</div>
    </div>

    <!-- ── ZONE 4: INFRA ── -->
    <div class="ar-zone z-infra">
      <div class="ar-zone-label">④ Infrastructure Tier</div>
      <div class="ar-infra-grid">

        <div class="ar-actor a-amber">
          <div class="ar-actor-head">
            <div class="ar-actor-icon">🧠</div>
            <div><div class="ar-actor-name">Redis</div><div class="ar-actor-role">Conversation memory</div></div>
          </div>
          <div class="ar-tags">
            <span class="ar-tag tag-store">🗃️ message history</span>
            <span class="ar-tag tag-store">🗃️ session TTL</span>
          </div>
        </div>

        <div class="ar-actor a-green">
          <div class="ar-actor-head">
            <div class="ar-actor-icon">🗃️</div>
            <div><div class="ar-actor-name">PostgreSQL</div><div class="ar-actor-role">Persistent store</div></div>
          </div>
          <div class="ar-tags">
            <span class="ar-tag tag-store">🗃️ threads</span>
            <span class="ar-tag tag-store">🗃️ messages</span>
          </div>
        </div>

        <div class="ar-actor a-sky">
          <div class="ar-actor-head">
            <div class="ar-actor-icon">📡</div>
            <div><div class="ar-actor-name">NATS</div><div class="ar-actor-role">Event bus</div></div>
          </div>
          <div class="ar-tags">
            <span class="ar-tag tag-recv">📬 worker events</span>
            <span class="ar-tag tag-send">📤 fanout all subs</span>
          </div>
        </div>

        <div class="ar-actor a-violet">
          <div class="ar-actor-head">
            <div class="ar-actor-icon">🤖</div>
            <div><div class="ar-actor-name">OpenAI API</div><div class="ar-actor-role">LLM inference</div></div>
          </div>
          <div class="ar-tags">
            <span class="ar-tag tag-recv">📬 messages array</span>
            <span class="ar-tag tag-send">📤 AssistantMessage</span>
          </div>
        </div>

      </div>
    </div>

  </div><!-- end zone grid -->

  <div class="ar-sep"></div>

  <!-- ════════ FLOWS ════════ -->
  <div class="ar-flows">

    <!-- NORMAL FLOW -->
    <div class="ar-flow-card fc-normal">
      <div class="ar-flow-title">✅ Normal Execution Flow — 11 steps</div>

      <div class="ar-step">
        <div class="ar-step-num">1</div>
        <div class="ar-step-body">
          <span class="ar-step-who">Browser</span>
          <div class="ar-step-text">User types message → POST /api/chat with threadId &amp; message</div>
        </div>
      </div>
      <div class="ar-step">
        <div class="ar-step-num">2</div>
        <div class="ar-step-body">
          <span class="ar-step-who">Next.js BFF</span>
          <div class="ar-step-text">Proxies request, opens persistent SSE pipe back to browser</div>
        </div>
      </div>
      <div class="ar-step">
        <div class="ar-step-num">3</div>
        <div class="ar-step-body">
          <span class="ar-step-who">FastAPI</span>
          <div class="ar-step-text">Registers SSE channel keyed by conversationId, calls RestateWorkflowClient</div>
        </div>
      </div>
      <div class="ar-step">
        <div class="ar-step-num">4</div>
        <div class="ar-step-body">
          <span class="ar-step-who">Restate</span>
          <div class="ar-step-text">Durably accepts workflow start, journals it, dispatches to Worker</div>
        </div>
      </div>
      <div class="ar-step">
        <div class="ar-step-num">5</div>
        <div class="ar-step-body">
          <span class="ar-step-who">Worker</span>
          <div class="ar-step-text">restore_memory() → loads full conversation history from Redis</div>
        </div>
      </div>
      <div class="ar-step">
        <div class="ar-step-num">6</div>
        <div class="ar-step-body">
          <span class="ar-step-who">OpenAI</span>
          <div class="ar-step-text">do_llm_call() → AssistantMessage returned, result journalled by Restate</div>
        </div>
      </div>
      <div class="ar-step">
        <div class="ar-step-num">7</div>
        <div class="ar-step-body">
          <span class="ar-step-who">Worker</span>
          <div class="ar-step-text">do_tool_exec() → tool runs, result journalled; loop repeats until end_turn</div>
        </div>
      </div>
      <div class="ar-step">
        <div class="ar-step-num">8</div>
        <div class="ar-step-body">
          <span class="ar-step-who">Redis + PG</span>
          <div class="ar-step-text">persist_message() → message written to both stores atomically</div>
        </div>
      </div>
      <div class="ar-step">
        <div class="ar-step-num">9</div>
        <div class="ar-step-body">
          <span class="ar-step-who">NATS</span>
          <div class="ar-step-text">publish_event() → text_delta / tool_result event fanned out to all subscribers</div>
        </div>
      </div>
      <div class="ar-step">
        <div class="ar-step-num">10</div>
        <div class="ar-step-body">
          <span class="ar-step-who">SSE Bridge</span>
          <div class="ar-step-text">WebHITLBridge receives NATS event, pushes to matching SSE channel</div>
        </div>
      </div>
      <div class="ar-step">
        <div class="ar-step-num">11</div>
        <div class="ar-step-body">
          <span class="ar-step-who">Browser</span>
          <div class="ar-step-text">EventSource fires, token/tool bubble rendered in real time ✨</div>
        </div>
      </div>
    </div>

    <!-- HITL FLOW -->
    <div class="ar-flow-card fc-hitl">
      <div class="ar-flow-title">⏸️ HITL Approval Flow — 7 steps</div>

      <div class="ar-step">
        <div class="ar-step-num">1</div>
        <div class="ar-step-body">
          <span class="ar-step-who">Worker</span>
          <div class="ar-step-text">Agent selects a tool with <strong>requires_approval=True</strong> (e.g. send_email)</div>
        </div>
      </div>
      <div class="ar-step">
        <div class="ar-step-num">2</div>
        <div class="ar-step-body">
          <span class="ar-step-who">Restate</span>
          <div class="ar-step-text"><code>ctx.promise("hitl-{id}")</code> created — Worker thread suspends, freed</div>
        </div>
      </div>
      <div class="ar-step">
        <div class="ar-step-num">3</div>
        <div class="ar-step-body">
          <span class="ar-step-who">NATS → Bridge</span>
          <div class="ar-step-text">tool_approval_request event published and pushed over SSE</div>
        </div>
      </div>
      <div class="ar-step">
        <div class="ar-step-num">4</div>
        <div class="ar-step-body">
          <span class="ar-step-who">Browser</span>
          <div class="ar-step-text">ToolApprovalCard rendered with tool name, inputs, and Approve / Reject buttons</div>
        </div>
      </div>
      <div class="ar-step">
        <div class="ar-step-num">5</div>
        <div class="ar-step-body">
          <span class="ar-step-who">Browser</span>
          <div class="ar-step-text">User clicks Approve → POST /api/chat/respond/{requestId}</div>
        </div>
      </div>
      <div class="ar-step">
        <div class="ar-step-num">6</div>
        <div class="ar-step-body">
          <span class="ar-step-who">FastAPI → Restate</span>
          <div class="ar-step-text">resolve_promise("hitl-{id}", approved:true) — Restate wakes the workflow instantly</div>
        </div>
      </div>
      <div class="ar-step">
        <div class="ar-step-num">7</div>
        <div class="ar-step-body">
          <span class="ar-step-who">Worker</span>
          <div class="ar-step-text">ctx.promise().get() returns approved value — tool executes from exact pause point</div>
        </div>
      </div>

      <!-- bonus: ask_human note -->
      <div style="margin-top:12px; padding:10px 12px; background:#1c1500; border-radius:8px; border:1px solid #92400e;">
        <div style="font-size:9px; font-weight:800; color:#fbbf24; letter-spacing:0.8px; text-transform:uppercase; margin-bottom:4px;">
          Same pattern for ask_human tool
        </div>
        <div style="font-size:10px; color:#a16207; line-height:1.5;">
          Instead of ToolApprovalCard → HumanInputCard shown. User types answer. Same promise resolution path. Worker receives text answer as ToolExecutionResultMessage.
        </div>
      </div>
    </div>

  </div><!-- end flows -->

  <!-- ════════ CRASH BANNER ════════ -->
  <div class="ar-crash">
    <div class="ar-crash-icon">🛡️</div>
    <div>
      <div class="ar-crash-title">Why Restate? — Crash-Safe by Construction</div>
      <div class="ar-crash-body">
        If the Worker process dies between any two activity steps, Restate <span class="hi">replays the durable journal</span> and resumes
        from the last committed result. The LLM is <span class="hi">never called twice</span>.
        Tool side-effects are <span class="hi">idempotency-keyed</span>. HITL promises survive restarts.
        Your conversation state is <span class="hi">never lost</span> — even during deploys.
      </div>
    </div>
  </div>

</div>
""")

In [9]:
from raavan.integrations.runtime.restate.workflows import (
    pipeline_workflow, chain_workflow, agent_workflow,
)

print("=== Registered Restate Workflows ===")
for wf in [pipeline_workflow, chain_workflow, agent_workflow]:
    print(f"\n  Workflow: {wf.name}")
    handlers = [h for h in dir(wf) if not h.startswith("_")]
    # show the run + any signal/query handlers
    for attr in ["run", "resolve_approval", "resolve_human_input", "status"]:
        fn = getattr(wf, attr, None)
        if fn is not None:
            print(f"    handler: {attr}")

=== Registered Restate Workflows ===

  Workflow: PipelineWorkflow

  Workflow: ChainWorkflow

  Workflow: AgentWorkflow


## 7. HITL — Human-In-The-Loop Promise Suspension

When the agent calls `ask_human` or a tool with `requires_approval=True`, the
workflow **suspends** by awaiting a Restate durable promise.  No polling, no
timeouts — Restate stores the pending promise and the worker thread is freed.

```
Worker                         Restate                       FastAPI / Frontend
  │                               │                               │
  │  ctx.promise("hitl-{id}")     │                               │
  ├──────────────────────────────►│                               │
  │  await promise.get()          │                               │
  │  [worker suspends]            │                               │
  │                               │  SSE: tool_approval_request   │
  │                               │──────────────────────────────►│
  │                               │                               │ user clicks Approve
  │                               │  POST /hitl/respond           │
  │                               │◄──────────────────────────────┤
  │                               │                               │
  │  resolve_promise("hitl-{id}") │                               │
  │                               │  promise resolved             │
  │  [worker resumes]             │◄──────────────────────────────┘
  │◄──────────────────────────────┤
  │  continue tool execution      │
```

The frontend hits `POST /api/chat/respond/{requestId}` → FastAPI calls
`workflow_client.resolve_promise(workflow_id, key, value)` → Restate wakes the workflow.

In [ ]:
from unittest.mock import AsyncMock, patch
from raavan.integrations.runtime.restate.client import RestateWorkflowClient
import asyncio

async def demo_hitl_flow():
    client = RestateWorkflowClient()

    # Simulate: worker calls resolve_approval (sent from the /hitl/respond route)
    with patch.object(client, "_invoke", new=AsyncMock(return_value=None)):
        await client.resolve_promise(
            workflow_name="AgentWorkflow",
            workflow_id="conv-001",
            promise_key="hitl-req-42",
            value={"approved": True, "comment": "Looks good"},
        )
        print("Promise resolved — workflow will resume in Restate")

    # Simulate: cancel a stuck workflow
    with patch.object(client, "_invoke", new=AsyncMock(return_value=None)):
        await client.cancel_workflow("AgentWorkflow", "conv-001")
        print("Workflow cancelled")

asyncio.run(demo_hitl_flow())

## 8. `PipelineWorkflow` — Chaining Adapter Steps

`PipelineWorkflow` executes a list of adapter steps sequentially.
Outputs can be forwarded between steps using `$prev.field` or `$step[n].field` references.

```python
steps = [
    {
        "adapter": "web_surfer",
        "inputs": {"url": "https://example.com/data.csv"},
    },
    {
        "adapter": "code_interpreter",
        "inputs": {
            "code": "import pandas as pd; df = pd.read_csv('$prev.file_path'); print(df.head())"
        },
    },
    {
        "adapter": "email_sender",
        "inputs": {
            "to": "analyst@corp.com",
            "subject": "Report ready",
            "body": "$step[1].stdout",
        },
    },
]

handle = await workflow_client.start_pipeline_workflow(
    pipeline_id="pipeline-daily-report",
    steps=steps,
    context={"user_id": "usr-001"},
)
```

Each step result is persisted by Restate. If step 2 crashes, the workflow
resumes from step 2 — `web_surfer` is **not** called again.

In [ ]:
from unittest.mock import AsyncMock, patch
from raavan.integrations.runtime.restate.client import RestateWorkflowClient
import asyncio

pipeline_steps = [
    {"adapter": "web_surfer",     "inputs": {"url": "https://httpbin.org/json"}},
    {"adapter": "code_interpreter","inputs": {"code": "print($prev.json_data)"}},
]

async def demo_pipeline():
    client = RestateWorkflowClient()
    with patch.object(client, "_invoke", new=AsyncMock(return_value={"wid": "pipe-001"})):
        handle = await client.start_pipeline_workflow(
            pipeline_id="pipe-001",
            steps=pipeline_steps,
            context={"user_id": "usr-demo"},
        )
    print("Pipeline started →", handle)

asyncio.run(demo_pipeline())

## 9. Runtime Summary & Selection Guide

| Runtime | When to use | Infrastructure needed |
|---|---|---|
| `LocalRuntime` | Notebooks, tests, single-process dev | None |
| `GrpcRuntime` | Production multi-tenant; durable ReAct loop | Restate cluster + NATS + Worker |
| `RestateRuntime` | Same as Grpc but wired via HTTP adapter | Restate cluster + NATS + Worker |

### Quick-start checklist — LocalRuntime (zero infra)
```python
from raavan.integrations.runtime.local import LocalRuntime
from raavan.core.agents.react_agent import ReActAgent

agent = ReActAgent(
    name="my-agent",
    model_client=llm,
    memory=UnboundedMemory(),
    runtime=LocalRuntime(),
    tools=[...],
)
response = await agent.run("Hello!")
```

### Quick-start checklist — Restate (durable)
1. `docker compose up restate nats postgres redis`
2. `python -m raavan.integrations.runtime.restate.worker &`
3. `uv run uvicorn raavan.server.app:app --port 8000`
4. Set `RESTATE_INGRESS_URL` + `RESTATE_ADMIN_URL` in `.env`
5. The server wires `app.state.workflow_client` in lifespan — all routes just use `request.app.state.workflow_client`

### Key env vars
```
RESTATE_INGRESS_URL=http://localhost:8080   # default
RESTATE_ADMIN_URL=http://localhost:9070    # default
```